# Scoring planners with `passage-eval` at 256×256

`passage-eval` is the evaluator of the benchmark. A planner receives the terrain, the obstacles and
the two endpoints; it never sees the reference route or its cost. The evaluator checks each returned
route, recomputes its cost and reports the six primary metrics of the paper. This notebook runs that
procedure at 256×256 on 32 validation tasks with three classical planners shipped with the
evaluator.

It is a demonstration of the scoring, not a benchmark result: the sample is small and the planners
are simple. **Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It solves a few dozen tasks and runs in about a minute, longer at 4096×4096.

In [ ]:
resolution = 256
split = "validation"
objective = "elevation"  # elevation, slope_uphill or energy
condition = "obstacles"  # obstacles or free
tasks = 32  # first validation scenes scored
seed = 0  # seed of the random-walk planner
output_dir = "notebook-artifacts/256x256/passage_eval_metrics"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2259dd94e2de28ab4a56c270247a7e347ec8f133"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))

## 1. The tasks

`scene_from_row` turns one released row into a task: elevation in metres, obstacle mask, endpoints,
and the reference route and cost that only the evaluator reads. Every released variant has the
status `ok`, so every scene gives a task with a reference route.

In [ ]:
from passage_eval.io.dataset import scene_from_row

SOLVER_OF = {"elevation": "astar_elevation", "slope_uphill": "astar_slope", "energy": "astar_energy"}
tasks_list, skipped = [], 0
for row in source.iter_rows(resolution, split, columns=["id", "tensor", "metadata"] + [
        f"path_astar_{s}_{condition}" for s in ("elevation", "slope", "energy")], samples=False):
    if row["metadata"]["paths"][SOLVER_OF[objective]][condition]["status"] != "ok":
        skipped += 1  # only an exact reference can be scored; every released variant is "ok"
        continue
    tasks_list.append(scene_from_row(row, resolution=resolution, cost_model=objective, split=split,
                                     task_variant=condition))
    if len(tasks_list) == tasks:
        break
overview = pd.DataFrame([{"Scene": t.sample_id, "Obstacle coverage (%)": 100 * float(np.mean(t.obstacles)),
                          "Reference cells": len(t.oracle_path), "Reference cost": t.oracle_cost}
                         for t in tasks_list])
display(Markdown(f"{len(tasks_list)} tasks · objective **{OBJECTIVE_LABELS[objective]}** · condition "
                 f"**{condition}** · median reference route {int(overview['Reference cells'].median())} cells"
                 + (f" · {skipped} variant(s) without an exact reference skipped." if skipped else ".")))

## 2. Three classical planners

- **A\*** uses the same grid search and weights as label generation. It should score a regret of zero
  and serves as a check that the evaluator and the labels agree.
- **Greedy** steps to the free neighbour closest to the goal and ignores the terrain cost. It is fast
  but pays for every hill it climbs, and it fails when obstacles trap it.
- **Random walk** takes 32 random legal steps, then lets A\* finish the route. It always connects,
  with a cost that depends on how far the random steps strayed.

In [ ]:
from passage_eval.baselines.classical import AStarPlanner, GreedyToGoalPlanner, RandomPolicyPlanner
from passage_eval.protocol.planner import evaluate_planner

PLANNER_COLORS = {"A*": "#238b45", "Greedy": "#1767ac", "Random walk": "#b8862b"}
planners = {"A*": AStarPlanner(), "Greedy": GreedyToGoalPlanner(), "Random walk": RandomPolicyPlanner(seed=seed)}
METRICS = ["cost_regret_median", "cost_regret_p90", "path_assd", "path_hausdorff_p95",
           "success_rate", "obstacle_violation_rate"]
results, per_task = [], {}
for name, planner in planners.items():
    metrics, arrays = evaluate_planner(planner, tasks_list)
    results.append({"Planner": name, **{m: metrics[m] for m in METRICS}})
    per_task[name] = arrays
scores = pd.DataFrame(results)
show(scores, f"Table 1. Six primary metrics on {len(tasks_list)} validation tasks at {resolution}×{resolution}.", "{:,.4f}")

Lower is better for every metric except `success_rate`. Regret is relative: 0.05 means a route 5 %
more expensive than the optimum. A route that fails to connect, or that crosses an obstacle, counts
with a regret of 999,999, so a single failure dominates the 90th percentile. The two geometric
metrics use only valid routes.

In [ ]:
fig, (ax_regret, ax_success) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.6, 1]})
fig.subplots_adjust(top=0.78, wspace=0.25)
for index, name in enumerate(planners):
    regret = np.asarray(per_task[name]["regret"], dtype=float)
    failed = ~np.asarray(per_task[name]["valid_path"], dtype=bool)  # censored: not connected or crossing
    shown = np.clip(100 * regret[~failed], 1e-3, None)
    jitter = np.linspace(-0.15, 0.15, num=shown.size)  # deterministic spread of the points
    ax_regret.scatter(shown, index + jitter, s=16, color=PLANNER_COLORS[name], alpha=0.75, linewidths=0)
    if failed.any():
        ax_regret.annotate(f"{int(failed.sum())} failed", (1.0, index), xycoords=("axes fraction", "data"),
                           xytext=(4, 0), textcoords="offset points", va="center", fontsize=8, color=MUTED)
ax_regret.set_xscale("log")
ax_regret.set_yticks(range(len(planners)), list(planners))
ax_regret.set(xlabel="Cost regret (%, log scale; zero regret drawn at 0.001 %)", title="Regret per task")
ax_regret.grid(True, axis="x")
ax_success.barh(list(planners), scores.success_rate * 100, color=[PLANNER_COLORS[p] for p in planners], height=0.5)
for y, value in enumerate(scores.success_rate * 100):
    ax_success.text(value + 1, y, f"{value:.0f} %", va="center", fontsize=8, color=INK)
ax_success.set(xlim=(0, 115), xlabel="Tasks connected (%)", title="Success rate")
ax_success.invert_yaxis()
ax_regret.invert_yaxis()
ax_success.grid(True, axis="x")
ax_success.grid(False, axis="y")
figure_title(fig, f"Three classical planners on {len(tasks_list)} validation tasks",
             f"{resolution}×{resolution} · {OBJECTIVE_LABELS[objective]} objective · {condition}")
figures = save_figure(fig, output_path, f"passage_eval_metrics-{resolution}")
plt.show()

*Figure 1. Left: cost regret of each task, on a log scale; tasks solved at the optimum are drawn at the
left edge, and failed tasks are counted at the right. Right: share of tasks where the planner connected
start and goal.*

## Summary

In [ ]:
s = scores.set_index("Planner")
display(Markdown("\n".join([
    f"- A\\* matches the stored labels: median regret {s.loc['A*', 'cost_regret_median']:.2g} and "
    f"success {100 * s.loc['A*', 'success_rate']:.0f} % on {len(tasks_list)} tasks.",
    f"- Greedy connects {100 * s.loc['Greedy', 'success_rate']:.0f} % of the tasks; its median regret is "
    f"{100 * s.loc['Greedy', 'cost_regret_median']:.1f} %.",
    f"- The random walk always connects through its A\\* closure; its median regret is "
    f"{100 * s.loc['Random walk', 'cost_regret_median']:.2f} %.",
    "- These numbers illustrate the evaluator on a small sample; a benchmark result needs the protocol of the "
    "dataset card: settings frozen on the calibration split, then one pass on the test split.",
])))
report = {"resolution": resolution, "split": split, "objective": objective, "condition": condition,
          "tasks": [t.sample_id for t in tasks_list], "scores": scores.to_dict("records"),
          "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"passage_eval_metrics-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*